# NEXUS Enhanced — GRPO Fine-Tuning on Qwen2.5-1.5B

**Team Falcons | Meta PyTorch OpenEnv Hackathon Grand Finale | April 25–26, 2026**

This notebook fine-tunes Qwen2.5-1.5B-Instruct as the Incident Commander (IC) agent
using Group Relative Policy Optimization (GRPO) on the NEXUS Enhanced environment.

## What this trains
The IC agent learns to:
- Prioritize correct alert signals and ignore red herrings
- Dispatch the right specialist agents at the right phase
- Cast correct coalition votes on medium+ incidents
- Send customer notifications early (L1 directive)
- Write detailed situation assessments (Mercor depth bonus)

## Environment
- 6 agents: IC (trained), L1, L2, SRE, PM (scripted), OversightAgent (monitor)
- 7 incidents: INC001 (easy) → INC007 (nightmare: CrowdStrike-scale)
- Reward: MTTR (30%) + Diagnosis (25%) + Customer (20%) + Coordination (15%) + Oversight (5%) + Depth bonus

## Notebook structure
1. Install dependencies
2. Connect to NEXUS environment server
3. Reward function + dataset
4. GRPO training (GPU required)
5. Evaluation: before/after comparison on INC003
6. Push checkpoint to HuggingFace Hub

---
## Cell 1 — Install Dependencies

In [ ]:
# Install Unsloth (fast 4-bit LoRA) + TRL GRPO + environment client
# Unsloth auto-detects CUDA version
!pip install -q unsloth
!pip install -q trl>=0.12.0 transformers>=4.45.0 accelerate>=0.34.0 peft>=0.13.0
!pip install -q datasets requests matplotlib

# If Unsloth install fails, uncomment the fallback:
# !pip install -q trl transformers accelerate peft bitsandbytes datasets requests matplotlib

print('Installation complete.')

---
## Cell 2 — Environment Setup

In [ ]:
import os, sys, json, re, time
import requests
from typing import Any, Dict, List, Optional

# ── Configure NEXUS server URL ─────────────────────────────────────────────
# Option A: HuggingFace Space (on-site deployment)
BASE_URL = os.environ.get("NEXUS_URL", "https://kunalkachru23-nexus-enhanced.hf.space")

# Option B: Local uvicorn (if running notebook alongside server)
# BASE_URL = "http://localhost:7860"

print(f"NEXUS server: {BASE_URL}")


# ── HTTP helpers ───────────────────────────────────────────────────────────
def _post(path, payload):
    r = requests.post(f"{BASE_URL}{path}", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

def _get(path):
    r = requests.get(f"{BASE_URL}{path}", timeout=30)
    r.raise_for_status()
    return r.json()


# ── Core environment interface ─────────────────────────────────────────────
def reset_env(incident_id=None, difficulty=None, seed=None):
    """Reset environment. Returns (session_id, observation)."""
    payload = {}
    if incident_id: payload["incident_id"] = incident_id
    if difficulty: payload["difficulty"] = difficulty
    if seed is not None: payload["seed"] = seed
    resp = _post("/reset", payload)
    return resp["session_id"], resp["observation"]

def step_env(session_id, action):
    """Execute IC action. Returns (observation, reward, done, info)."""
    resp = _post(f"/step/{session_id}", action)
    return resp["observation"], resp["reward"], resp["done"], resp.get("info", {})

def get_state(session_id):
    return _get(f"/state/{session_id}")

def get_reward(session_id):
    return _get(f"/reward/{session_id}")


# ── Baseline scripted policy (fills in remaining steps during rollouts) ────
def baseline_policy(obs):
    """Deterministic scripted IC — used to complete episodes after model's first action."""
    phase = obs.get("phase", "detection")
    step = obs.get("step", 0)
    competing = obs.get("competing_hypotheses", [])
    coalition_vote = competing[-1] if competing and step >= 10 else None
    resolution_confidence = 0.0
    if phase in ("resolution", "postmortem") or step > 18:
        resolution_confidence = min(0.9, 0.05 * step)
    return {
        "situation_assessment": f"Baseline policy: phase={phase}, step={step}. Continuing investigation.",
        "hypothesis": "Root cause under investigation",
        "coalition_vote": coalition_vote,
        "l1_directive": {"action": "check_customer_reports", "parameters": {}, "reasoning": "Monitor impact"},
        "l2_directive": {"action": "check_all_alerts", "parameters": {}, "reasoning": "Sweep alerts"},
        "sre_directive": {"action": "execute_runbook_step", "parameters": {"step_id": "rb_check_logs"}, "reasoning": "Follow runbook"},
        "pm_directive": {"action": "track_revenue_impact", "parameters": {}, "reasoning": "Track SLA"},
        "resolution_confidence": resolution_confidence,
        "escalation_required": step > 8,
    }


# ── Verify connection ──────────────────────────────────────────────────────
try:
    health = _get("/health")
    print(f"Server health: {health}")

    session_id, obs = reset_env("INC003")
    print(f"\nTest reset OK:")
    print(f"  Incident : {obs['incident_id']} — {obs.get('incident_title', '')}")
    print(f"  Severity : {obs.get('severity', '').upper()}")
    print(f"  Alerts   : {len(obs.get('initial_alerts', []))}")
    print(f"  Hyp count: {len(obs.get('competing_hypotheses', []))}")
    print(f"\nEnvironment ready. session_id={session_id[:8]}...")
except Exception as e:
    print(f"⚠ Server connection failed: {e}")
    print(f"  Make sure NEXUS server is running at {BASE_URL}")
    print("  For local: uvicorn server.app:app --port 7860 --reload")

---
## Cell 3 — Prompt Builder, Reward Function, and Dataset

In [ ]:
# ── IC Prompt Builder ──────────────────────────────────────────────────────

IC_SYSTEM_PROMPT = """You are the Incident Commander (IC) for NEXUS, a production incident response AI.
Coordinate 4 specialist agents (L1 Support, L2 Engineer, SRE, Product Manager) to resolve incidents.

Expert review criteria: {expert_criteria}
- speed: prioritize MTTR | communication: prioritize customer notifications
- technical: prioritize root cause accuracy | cost: minimize duplicate tool calls

RESPONSE: JSON only.
{{
  "situation_assessment": "<detailed multi-sentence analysis>",
  "hypothesis": "<specific root cause hypothesis>",
  "coalition_vote": "<hypothesis vote if in investigation phase, else null>",
  "l1_directive": {{"action": "<action>", "parameters": {{}}, "reasoning": "<why>"}},
  "l2_directive": {{"action": "<action>", "parameters": {{}}, "reasoning": "<why>"}},
  "sre_directive": {{"action": "<action>", "parameters": {{}}, "reasoning": "<why>"}},
  "pm_directive": {{"action": "<action>", "parameters": {{}}, "reasoning": "<why>"}},
  "resolution_confidence": 0.0,
  "escalation_required": false
}}

Agent actions — L1: check_customer_reports, send_notification, get_sla_status
L2: check_all_alerts, query_metric, query_logs, check_deploy_history
SRE: execute_runbook_step, list_runbooks  |  PM: track_revenue_impact, get_breach_risk"""


def build_ic_prompt(obs):
    expert_criteria = obs.get("expert_criteria", "technical")
    system = IC_SYSTEM_PROMPT.format(expert_criteria=expert_criteria)

    alerts = obs.get("initial_alerts", [])
    alert_lines = "\n".join(
        f"  [{a.get('status','?'):8s}] {a.get('service')}.{a.get('metric')} = {a.get('value')}"
        for a in alerts
    ) or "  (none)"

    findings = obs.get("agent_findings", [])
    finding_lines = "\n".join(
        f"  [{f.get('agent'):12s}] {f.get('finding')}"
        for f in findings[-6:]
    ) or "  (none yet)"

    competing = obs.get("competing_hypotheses", [])
    hyp_block = ""
    if competing:
        hyp_block = "\nCOMPETING HYPOTHESES:\n" + "\n".join(f"  [{i+1}] {h}" for i, h in enumerate(competing))

    user_msg = (
        f"INCIDENT: [{obs.get('incident_id')}] {obs.get('incident_title','')}\n"
        f"SEVERITY: {obs.get('severity','').upper()} | PHASE: {obs.get('phase','').upper()} "
        f"| STEP: {obs.get('step',0)} | ELAPSED: {obs.get('elapsed_minutes',0):.0f}min\n"
        f"SERVICES: {', '.join(obs.get('affected_services',[]))}\n"
        f"CUSTOMER REPORTS: {'; '.join(obs.get('customer_reports',[])[:2])}\n\n"
        f"ACTIVE ALERTS:\n{alert_lines}\n\n"
        f"AGENT FINDINGS:\n{finding_lines}\n"
        f"{hyp_block}\n"
        f"NOTIFICATIONS SENT: {obs.get('notifications_sent', 0)} | "
        f"OVERSIGHT VIOLATIONS: {obs.get('oversight_violations', 0)}\n\n"
        f"Respond with your IC action JSON:"
    )

    return (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


# ── Action parser ──────────────────────────────────────────────────────────

def parse_ic_action(completion, obs=None):
    """Parse model completion → action dict. Never raises."""
    defaults = {"action": "no_op", "parameters": {}, "reasoning": ""}

    def _extract_json(text):
        text = text.strip()
        try:
            return json.loads(text)
        except Exception:
            pass
        start = text.find("{")
        if start == -1:
            return {}
        depth = 0
        for i, ch in enumerate(text[start:], start):
            if ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try: return json.loads(text[start:i+1])
                    except: break
        return {}

    def _dir(key, raw):
        val = raw.get(key)
        return {**defaults, **val} if isinstance(val, dict) else dict(defaults)

    raw = _extract_json(completion)
    conf = float(raw.get("resolution_confidence", 0.0))
    conf = max(0.0, min(1.0, conf))

    return {
        "situation_assessment": str(raw.get("situation_assessment", "")),
        "hypothesis": str(raw.get("hypothesis", "")),
        "coalition_vote": raw.get("coalition_vote"),
        "l1_directive": _dir("l1_directive", raw),
        "l2_directive": _dir("l2_directive", raw),
        "sre_directive": _dir("sre_directive", raw),
        "pm_directive": _dir("pm_directive", raw),
        "resolution_confidence": conf,
        "escalation_required": bool(raw.get("escalation_required", False)),
    }


# ── Reward function (TRL-compatible) ──────────────────────────────────────

INCIDENT_POOL = {
    "easy": ["INC001", "INC002"],
    "medium": ["INC003", "INC004"],
    "hard": ["INC005", "INC006"],
}

def reward_fn(prompts, completions, **kwargs):
    """
    TRL GRPO reward function.
    For each (prompt, completion):
      1. Extract incident_id from prompt
      2. Reset environment
      3. Execute completion as step-1 action
      4. Complete episode with baseline policy
      5. Return final episode reward
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        try:
            # TRL GRPO passes completions as ['text'] lists — unwrap if needed
            if isinstance(completion, list):
                completion = completion[0] if completion else ""

            # Extract incident from prompt
            m = re.search(r"\[(INC\d+)\]", prompt)
            incident_id = m.group(1) if m else "INC003"

            # Reset fresh episode
            session_id, obs = reset_env(incident_id=incident_id)

            # Model's first action
            action = parse_ic_action(completion, obs)
            obs, reward, done, info = step_env(session_id, action)

            # Complete with baseline (cap at 15 steps for training speed)
            step = 1
            while not done and step < 15:
                obs, reward, done, info = step_env(session_id, baseline_policy(obs))
                step += 1

            if not done:
                partial = get_reward(session_id)
                episode_reward = partial.get("total", 0.0)
            else:
                episode_reward = reward

            rewards.append(float(episode_reward))

        except Exception as e:
            print(f"  [reward_fn] rollout error: {e}")
            rewards.append(0.0)

    return rewards


# ── Build GRPO dataset ─────────────────────────────────────────────────────

from datasets import Dataset

N_PROMPTS = 100  # Adjust based on available time
TRAIN_DIFFICULTIES = ["easy", "medium"]  # Start easy; add "hard" after step 100

print("Building training dataset...")
prompt_records = []
for i in range(N_PROMPTS):
    difficulty = TRAIN_DIFFICULTIES[i % len(TRAIN_DIFFICULTIES)]
    incidents = INCIDENT_POOL[difficulty]
    incident_id = incidents[i % len(incidents)]
    try:
        sid, obs = reset_env(incident_id=incident_id)
        prompt_records.append({"prompt": build_ic_prompt(obs)})
    except Exception as e:
        print(f"  Skipping {incident_id}: {e}")

train_dataset = Dataset.from_list(prompt_records)
print(f"Dataset ready: {len(train_dataset)} prompts")

# Quick reward function smoke test
print("\nSmoke-testing reward function (1 rollout)...")
test_prompt = prompt_records[0]["prompt"]
test_completion = json.dumps({
    "situation_assessment": "Investigating INC003. Memory pressure on recommendation-service. Directing L2 to check logs.",
    "hypothesis": "Memory leak in recommendation-service ML model",
    "coalition_vote": None,
    "l1_directive": {"action": "send_notification", "parameters": {}, "reasoning": "Customer notification"},
    "l2_directive": {"action": "query_logs", "parameters": {"service": "recommendation-service"}, "reasoning": "Check error logs"},
    "sre_directive": {"action": "list_runbooks", "parameters": {}, "reasoning": "Enumerate options"},
    "pm_directive": {"action": "track_revenue_impact", "parameters": {}, "reasoning": "SLA check"},
    "resolution_confidence": 0.1,
    "escalation_required": True,
})
# Test both plain string and list-wrapped (as TRL sends it)
test_rewards_plain = reward_fn([test_prompt], [test_completion])
test_rewards_wrapped = reward_fn([test_prompt], [[test_completion]])
print(f"Smoke test reward (plain):   {test_rewards_plain[0]:.4f}")
print(f"Smoke test reward (wrapped): {test_rewards_wrapped[0]:.4f}")
assert test_rewards_plain[0] == test_rewards_wrapped[0], "List unwrapping mismatch!"
print("Reward function ready.")

---
## Cell 4 — GRPO Training

**Requires GPU (T4 or better).** Runtime → Change runtime type → GPU.

In [ ]:
import torch

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_STEPS = 200          # ~2-3h on T4; reduce to 50 for quick validation
GROUP_SIZE = 8         # Completions per prompt per step
LEARNING_RATE = 5e-6
OUTPUT_DIR = "/content/nexus_grpo_checkpoint"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"Training: {MODEL_ID} for {N_STEPS} steps, G={GROUP_SIZE}")

# ── Load model via Unsloth (4-bit LoRA) ───────────────────────────────────
try:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("Loaded via Unsloth (4-bit LoRA, r=16)")

except ImportError:
    print("Unsloth not available — using standard transformers + PEFT")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, LoraConfig, TaskType

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )
    lora_config = LoraConfig(
        r=16, lora_alpha=16,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.0, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Configure GRPOTrainer ──────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=N_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=max(1, N_STEPS // 20),
    lr_scheduler_type="cosine",
    optim="adamw_8bit",

    # GRPO-specific
    num_generations=GROUP_SIZE,
    max_completion_length=512,
    max_prompt_length=1024,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,  # KL penalty

    # Logging / saving
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    seed=42,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# ── Train ──────────────────────────────────────────────────────────────────
print(f"\nStarting GRPO training — {N_STEPS} steps, G={GROUP_SIZE}...")
print("(Screenshot this output at steps 0, 25, 50, 100, 150 for the pitch)")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\nTraining complete: {elapsed/60:.1f} minutes")

# Save checkpoint
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Checkpoint saved → {OUTPUT_DIR}")

---
## Cell 5 — Evaluation: Before/After Comparison on INC003

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Run trained model on INC003 ────────────────────────────────────────────
try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
except ImportError:
    pass  # Standard transformers handles inference normally

def generate_ic_action(obs, max_new_tokens=512):
    prompt = build_ic_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_ic_action(completion, obs)


def run_evaluation_episode(incident_id="INC003", use_model=True, seed=42):
    """Run one evaluation episode. Returns reward breakdown."""
    sid, obs = reset_env(incident_id=incident_id, seed=seed)
    done = False
    step = 0
    reward = 0.0
    actions_taken = []

    while not done and step < 25:
        if use_model and step == 0:  # Model takes first action; baseline fills rest
            action = generate_ic_action(obs)
        else:
            action = baseline_policy(obs)

        actions_taken.append({
            "step": step + 1,
            "hypothesis": action.get("hypothesis", "")[:80],
            "coalition_vote": action.get("coalition_vote"),
            "resolution_confidence": action.get("resolution_confidence", 0),
        })

        obs, reward, done, info = step_env(sid, action)
        step += 1

    state = get_state(sid)
    rb = state.get("reward_breakdown") or {}
    return {
        "total": rb.get("total", 0.0),
        "mttr": rb.get("mttr", 0.0),
        "diagnosis": rb.get("diagnosis", 0.0),
        "customer": rb.get("customer", 0.0),
        "coordination": rb.get("coordination", 0.0),
        "oversight": rb.get("oversight", 0.0),
        "depth_bonus": rb.get("depth_bonus", 0.0),
        "coalition_correct": state.get("coalition_correct"),
        "notifications_sent": state.get("notifications_sent", 0),
        "steps": step,
        "elapsed_minutes": state.get("elapsed_minutes", 0),
        "actions": actions_taken,
    }


print("Running evaluation on INC003 (5 episodes each)...")
N_EVAL = 5

baseline_results = [run_evaluation_episode(use_model=False, seed=i) for i in range(N_EVAL)]
trained_results = [run_evaluation_episode(use_model=True, seed=i) for i in range(N_EVAL)]

def avg(results, key):
    vals = [r[key] for r in results if r.get(key) is not None]
    return sum(vals) / max(len(vals), 1)

print("\n" + "="*55)
print("  Before/After Comparison — INC003 (Memory Leak)")
print("="*55)
print(f"  {'Metric':<22} {'Baseline':>10} {'Trained':>10} {'Delta':>8}")
print("-"*55)
for key in ["total", "mttr", "diagnosis", "customer", "coordination", "depth_bonus"]:
    b = avg(baseline_results, key)
    t = avg(trained_results, key)
    print(f"  {key:<22} {b:>10.4f} {t:>10.4f} {t-b:>+8.4f}")
b_notif = avg(baseline_results, "notifications_sent")
t_notif = avg(trained_results, "notifications_sent")
print(f"  {'notifications_sent':<22} {b_notif:>10.1f} {t_notif:>10.1f} {t_notif-b_notif:>+8.1f}")
print("="*55)


# ── Plot before/after chart ────────────────────────────────────────────────
dims = ["mttr", "diagnosis", "customer", "coordination", "oversight", "depth_bonus"]
labels = ["MTTR\n30%", "Diagnosis\n25%", "Customer\n20%", "Coord\n15%", "Oversight\n5%", "Depth\nBonus"]
colors_b = ["#aec6cf"] * 6
colors_t = ["#4C72B0"] * 6

b_vals = [avg(baseline_results, d) for d in dims]
t_vals = [avg(trained_results, d) for d in dims]

x = range(len(dims))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Reward breakdown comparison
bars_b = ax1.bar([i - width/2 for i in x], b_vals, width, label="Baseline", color="#aec6cf", edgecolor="white")
bars_t = ax1.bar([i + width/2 for i in x], t_vals, width, label="Trained (GRPO)", color="#4C72B0", edgecolor="white")
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.set_ylabel("Score (0–1)", fontsize=11)
ax1.set_title("INC003: Reward Breakdown — Baseline vs Trained", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(axis="y", alpha=0.3)
for bar in bars_t:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
             f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8, color="#333")

# Total reward comparison
b_totals = [r["total"] for r in baseline_results]
t_totals = [r["total"] for r in trained_results]
ax2.plot(range(1, N_EVAL+1), b_totals, "o--", color="#aec6cf", linewidth=2, markersize=8, label="Baseline")
ax2.plot(range(1, N_EVAL+1), t_totals, "o-", color="#4C72B0", linewidth=2, markersize=8, label="Trained (GRPO)")
ax2.axhline(y=sum(b_totals)/len(b_totals), color="#aec6cf", linestyle=":", linewidth=1)
ax2.axhline(y=sum(t_totals)/len(t_totals), color="#4C72B0", linestyle=":", linewidth=1)
ax2.set_xlabel("Evaluation Episode", fontsize=11)
ax2.set_ylabel("Total Reward", fontsize=11)
ax2.set_title("INC003: Total Reward per Episode", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/nexus_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → /content/nexus_before_after.png")

---
## Cell 6 — Push Checkpoint to HuggingFace Hub

In [ ]:
from huggingface_hub import login, HfApi

# ── Authenticate ───────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Set via Colab Secrets or os.environ
HF_USERNAME = os.environ.get("HF_USERNAME", "team-falcons")
REPO_ID = f"{HF_USERNAME}/nexus-qwen-1.5b-grpo"

if HF_TOKEN:
    login(token=HF_TOKEN)
    print(f"Logged in to HuggingFace as {HF_USERNAME}")
else:
    print("HF_TOKEN not set — run: login(token='YOUR_TOKEN')")

# ── Push model ─────────────────────────────────────────────────────────────
try:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    model.push_to_hub(REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

    # Upload evaluation chart
    api = HfApi()
    api.upload_file(
        path_or_fileobj="/content/nexus_before_after.png",
        path_in_repo="nexus_before_after.png",
        repo_id=REPO_ID,
        token=HF_TOKEN,
    )

    print(f"\nModel pushed → https://huggingface.co/{REPO_ID}")
    print(f"Chart uploaded → https://huggingface.co/{REPO_ID}/blob/main/nexus_before_after.png")

except Exception as e:
    print(f"Push failed: {e}")
    print("Checkpoint is saved locally at:", OUTPUT_DIR)